# A demo of a pipeline for TargetTiller

This demo uses the `FilesHelper` class to locate files to load. (Checkout the cell that runs the help for the class below.)

The demo runs all three types of balancing supported.

In [ ]:
import sys

sys.path.append("../..")

from target_tiller import TransactionsCalculator
from target_tiller.input import (
    FilesHelper,
    PortfolioLoaderYAML,
    TargetsLoaderYAML,
    TDPortfolioLoaderCSV,
    TransactionsConfigsLoaderYAML,
)
from target_tiller.output import ReportJupyter

# Take the default files to run.
# Without further configuration, these will be the demo files.
files_helper = FilesHelper()

# The defaults can be overridden via environment variables,
# or through arguments to direct the behavior, e.g.,
# files_helper = FilesHelper("demo")

def main() -> None:
    targets_loader = TargetsLoaderYAML(files_helper.targets_file)
    transactions_configs_loader = \
      TransactionsConfigsLoaderYAML(files_helper.transactions_configs_file)

    targets = targets_loader.targets
    transactions_configs = transactions_configs_loader.transactions_configs

    # Hash the portfolios by name because the transactions_configs
    # needs to find them by name
    portfolio_loaders = {}
    for filename in files_helper.portfolio_files:
        if filename[-4:] == ".csv":
            portfolio_loader = TDPortfolioLoaderCSV(filename)
        else:
            portfolio_loader = PortfolioLoaderYAML(filename)
        portfolio_loaders[portfolio_loader.name] = portfolio_loader

    reporter = ReportJupyter()

    for transactions_config in transactions_configs:
        portfolio_loader = portfolio_loaders[transactions_config.portfolio_name]
        portfolio = portfolio_loader.portfolio
        transactions_calculator = \
        TransactionsCalculator(portfolio=portfolio,
                               targets=targets,
                               transactions_config=transactions_config).run()

        reporter.report(transactions_calculator,
                        portfolio_filename=portfolio_loader.filename)

main()

## Which files were used for the demo?

The following cell shows the files and the contents used in the demo.
The next cell after shows some help for configuring the files used.

In [ ]:
files_helper.report()

In [ ]:
help(files_helper)

# Another demo ...

This demo doesn't use any of the file loaders, nor does it rely on the fancy demo stuff, it just works on the objects configured via python.

We just run this for a single portfolio for simplicity, This is likely *not* how you want to do things in practice, but might help you understand how the pipeline works.

This demo produces both an HTML report and a JSON one.

In [ ]:
import sys

sys.path.append("../..")

from target_tiller import Portfolio, Targets, TransactionsCalculator, TransactionsConfig
from target_tiller.output import ReportJSON, ReportJupyter


def main() -> None:
    targets = Targets({"LOL": 0.2,
                       "HUH": 0.5,
                       "WTF": 0.3})

    portfolio = Portfolio(holdings={"LOL": 2000,
                                    "WHY": 100,
                                    "HUH": 6000,
                                    "WTF": 2000},
                          cash=10000,
                          name="123")

    transactions_config = \
      TransactionsConfig(portfolio_name="123",
                         nickname="nest egg",
                         action="buy_only",
                         amount=1500,
                         minimum_purchase=500,
                         maximum_purchase=1000)

    transactions_calculator = \
      TransactionsCalculator(portfolio=portfolio,
                             targets=targets,
                             transactions_config=transactions_config).run()

    print("HTML/Jupyter report")
    reporter = ReportJupyter()
    reporter.report(transactions_calculator)

    print("JSON report")
    reporter = ReportJSON()
    reporter.report(transactions_calculator)

main()
